# Time-series transformer for regression — end-to-end demo

```
time series (B, C, T)
   -> Conv1d local feature extractor / patch tokenizer      (B, C, N, d)
   -> systematic-missing embedding  (additive | concat)
   -> transformer, attention ACROSS FEATURES                (B, C, N, d)
   -> masked pooling  ->  time-series embedding             (B, d_ts)
   -> LATE FUSION with the snapshot embedding               (B, d_ts + d_snap)
   -> regression head                                       (B, n_targets)
```

This notebook walks through the whole pipeline on synthetic credit-card-like data:
three tables joined on a primary key, two kinds of missingness, train / holdout /
out-of-time splits, and per-epoch RMSE / MAE / R² / NMAD logging on all three splits.

**Swap in your own data at step 2** — everything downstream is unchanged.

## 0. Setup

In [ ]:
import sys, pathlib

# make `import tsfm` work no matter where the notebook is launched from
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / "tsfm" / "__init__.py").exists():
        sys.path.insert(0, str(_p)); break

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

from tsfm import (
    DataConfig, TokenizerConfig, MissingConfig, TransformerConfig, SnapshotConfig,
    HeadConfig, TrainConfig, TSFormerRegressor, Trainer, prepare_datasets, set_seed,
)
from tsfm.data import SystematicMissingSpec
from tsfm.simulate import make_synthetic_panel

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

## 1. The three input tables

| table | grain | contents |
|---|---|---|
| `ts_long` | one row per (key, time step) | the `C` time-series features |
| `snapshot` | one row per key | static categoricals + numerics (incl. tenure) |
| `target` | one row per key | the regression target(s) |

`month_idx = 0` is the **oldest** month here (`time_order="oldest_first"`).
If your panel is stored newest-first (`bal_m1` = last month), set
`time_order="newest_first"` and the systematic-missing logic flips accordingly.

In [ ]:
N_FEATURES, SEQ_LEN = 8, 24
ts_long, snapshot, target, splits = make_synthetic_panel(
    n_records=8000, n_features=N_FEATURES, seq_len=SEQ_LEN,
    random_missing_rate=0.05, seed=42,
)
FEATURES = [f"f{i}" for i in range(N_FEATURES)]
CAT_COLS = ["cat0", "cat1", "cat2"]
NUM_COLS = ["num0", "num1", "num2", "num3", "tenure_months"]

print("ts_long   ", ts_long.shape); display(ts_long.head(3))
print("snapshot  ", snapshot.shape); display(snapshot.head(3))
print("target    ", target.shape); display(target.head(3))
print({k: len(v) for k, v in splits.items()})

### The two kinds of missingness

* **Systematic** — the month *cannot* exist: a card booked 5 months ago has no
  data for the other 19 months of a 24-month window. This is a signal about the
  record, so it gets a **learned missing embedding**.
* **Random** — the month exists but the value was not recorded. Imputed with `0`
  (the training mean after scaling) and flagged to the conv through an extra
  observed-mask channel.

In [ ]:
piv = ts_long.pivot(index="account_id", columns="month_idx", values="f0")
order = snapshot.set_index("account_id").loc[piv.index, "tenure_months"].sort_values().index
fig, ax = plt.subplots(figsize=(9, 4))
ax.imshow(piv.loc[order[:400]].isna().to_numpy(), aspect="auto", cmap="Greys", interpolation="nearest")
ax.set_xlabel("month_idx (0 = oldest)"); ax.set_ylabel("accounts, sorted by tenure")
ax.set_title("missingness of f0 — the triangle is systematic, the speckle is random")
plt.tight_layout(); plt.show()

## 2. Data processing

`DataConfig` names the columns; `SystematicMissingSpec` says how to separate the
two kinds of missingness:

* `method="tenure"` — best when you have a months-on-book column (used here);
* `method="boundary_nan"` — infer it from an unbroken run of all-NaN steps at the
  start (oldest-first) or end (newest-first) of the window;
* `method="explicit"` — you pass the mask yourself;
* `method="none"` — treat everything as random.

Scalers, categorical vocabularies and the target scaler are fitted on **train only**.

In [ ]:
data_cfg = DataConfig(
    key="account_id",
    time_col="month_idx",
    ts_features=FEATURES,
    target_cols=["y"],
    snapshot_cat=CAT_COLS,
    snapshot_num=NUM_COLS,
    time_order="oldest_first",
    systematic=SystematicMissingSpec(method="tenure", tenure_col="tenure_months",
                                     tenure_unit_steps=1.0),
    ts_scaler="standard",      # fitted on observed train cells only
    target_scaler="standard",  # metrics are always reported in original units
    clip_sigma=10.0,
)

prep = prepare_datasets(ts_long, snapshot, target, data_cfg, splits)
prep.summary()

In [ ]:
# What one batch looks like
from tsfm import make_dataloader
batch = next(iter(make_dataloader(prep.datasets["train"], batch_size=4, pin_memory=False)))
for k, v in batch.items():
    print(f"{k:12s} {tuple(v.shape)}  {v.dtype}")

## 3. Build the model

Every knob the architecture needs is a hyper-parameter:

* **tokenizer** — `conv_channels` (out-channels per feature), `kernel_size`,
  `stride`, `n_layers`, `d_model`. `per_feature=True` makes the conv *grouped*,
  so each feature keeps its own filters and every token stays attributable to one
  feature — which is what makes feature-axis attention meaningful.
* **missing** — `combine="add"` or `"concat"` (the switch you asked for).
* **transformer** — `attn_mode="feature"` attends **across features** at each
  patch position; the Conv1d has already handled the temporal pattern.
* **snapshot / head** — static branch and the late-fusion MLP.

In [ ]:
model_cfg = prep.model_config(
    tokenizer=TokenizerConfig(
        in_channels=prep.n_features,
        conv_channels=32,      # conv out-channels PER feature
        kernel_size=5,
        stride=2,
        n_layers=1,
        d_model=64,
        per_feature=True,      # grouped conv -> one token stream per feature
        pad_to_cover=True,     # never silently drop the last (most recent) month
        norm="batch",
        dropout=0.0,
    ),
    missing=MissingConfig(
        combine="add",                 # <- "add" or "concat"
        d_missing=16,                  # only used by "concat" ("add" forces d_model)
        soft=True,                     # scale by the missing FRACTION inside the patch
        per_feature_embedding=True,
        add_observed_channel=True,     # random missingness reaches the conv too
        mask_fully_missing_tokens=True,
    ),
    transformer=TransformerConfig(
        n_layers=4, n_heads=8, d_ff=256, dropout=0.1,
        attn_mode="feature",           # attention ACROSS FEATURES
        feature_embedding=True, patch_embedding=True,
        pool="attention",
    ),
    snapshot=SnapshotConfig(hidden_dims=(64,), d_out=32, dropout=0.1),
    head=HeadConfig(hidden_dims=(128, 64), dropout=0.1),
)

model = TSFormerRegressor(model_cfg)
pd.Series(model.describe())

## 4. Train

Model selection is on **lowest holdout RMSE**. OOT is scored every epoch but never
influences selection or early stopping — it only tells you how much you lose out of time.

In [ ]:
train_cfg = TrainConfig(
    epochs=30,
    batch_size=256,
    lr=1e-3,
    weight_decay=1e-2,
    loss="mse",
    scheduler="cosine",
    warmup_epochs=2,
    grad_clip=1.0,
    amp=True, amp_dtype="bf16",       # mixed precision on GPU; ignored on CPU
    early_stopping_patience=8,
    monitor="rmse",                   # selection metric on holdout
    num_workers=4 if DEVICE == "cuda" else 0,
    pin_memory=DEVICE == "cuda",
    device=DEVICE,
    seed=42,
    ckpt_path="best_model.pt",
)

trainer = Trainer(model, train_cfg, target_scaler=prep.target_scaler,
                  target_names=prep.target_names)
history = trainer.fit(prep.datasets["train"], prep.datasets["holdout"], prep.datasets["oot"])

In [ ]:
history.head()

## 5. Metrics: RMSE / MAE / R² / NMAD on all three splits

In [ ]:
report = trainer.final_report(prep.datasets)
display(report.style.format("{:.4f}", subset=["rmse", "mae", "r2", "nmad"]))
print(f"selected epoch {trainer.best_epoch_} (lowest holdout RMSE)")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
for ax, metric in zip(axes, ["rmse", "mae", "r2", "nmad"]):
    for split, style in [("train", "-"), ("holdout", "-"), ("oot", "--")]:
        col = f"{split}_{metric}"
        if col in history:
            ax.plot(history.index, history[col], style, label=split)
    ax.axvline(trainer.best_epoch_, color="grey", lw=0.8, ls=":")
    ax.set_title(metric.upper()); ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
axes[0].legend()
plt.suptitle("per-epoch metrics (dotted line = selected epoch)", y=1.05)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, split in zip(axes, ["train", "holdout", "oot"]):
    yt, yp = trainer.predict(prep.datasets[split])
    ax.scatter(yt, yp, s=4, alpha=0.25)
    lo, hi = float(min(yt.min(), yp.min())), float(max(yt.max(), yp.max()))
    ax.plot([lo, hi], [lo, hi], "r--", lw=1)
    ax.set_title(f"{split}  (n={len(yt)})"); ax.set_xlabel("actual"); ax.set_ylabel("predicted")
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Residuals: RMSE is driven by the tails, NMAD by the core of the distribution
yt, yp = trainer.predict(prep.datasets["oot"])
resid = (yt - yp).ravel()
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.hist(resid, bins=60)
ax.set_title(f"OOT residuals — rmse {np.sqrt((resid**2).mean()):.2f}, "
             f"nmad {1.4826*np.median(np.abs(resid-np.median(resid))):.2f}")
ax.set_xlabel("actual - predicted"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Does performance depend on tenure?

Short-tenure accounts are the ones the missing embedding has to carry. Slicing the
error by tenure tells you whether that part of the model is doing its job.

In [ ]:
oot = prep.datasets["oot"]
yt, yp = trainer.predict(oot)
ten = snapshot.set_index("account_id").loc[oot.ids, "tenure_months"].to_numpy()
bins = pd.cut(ten, [0, 6, 12, 18, 24], labels=["1-6m", "7-12m", "13-18m", "19-24m"])
by_tenure = (pd.DataFrame({"bucket": bins, "err": (yt - yp).ravel()})
             .groupby("bucket", observed=True)["err"]
             .agg(n="size", rmse=lambda e: float(np.sqrt((e**2).mean())),
                  mae=lambda e: float(np.abs(e).mean())))
display(by_tenure)

## 6. What the model attends to

The pooling layer returns its attention weights, so you can see which
(feature, patch) tokens drive the embedding. Averaged over records this is a cheap
sanity check — not a causal explanation.

In [ ]:
model.eval()
b = next(iter(make_dataloader(prep.datasets["holdout"], batch_size=512, pin_memory=False)))
b = {k: v.to(trainer.device) for k, v in b.items()}
with torch.no_grad():
    _, aux = model(b, return_aux=True)

w = aux["pool_attn"].float().cpu().numpy()          # (B, C, N)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].bar(FEATURES, w.sum(axis=2).mean(axis=0))
axes[0].set_title("pooling attention by feature"); axes[0].tick_params(axis="x", rotation=45)
im = axes[1].imshow(w.mean(axis=0), aspect="auto", cmap="viridis")
axes[1].set_yticks(range(len(FEATURES)), FEATURES)
axes[1].set_xlabel("patch (0 = oldest)"); axes[1].set_title("mean attention over the token grid")
plt.colorbar(im, ax=axes[1]); plt.tight_layout(); plt.show()

## 7. Ablation: additive vs concatenation missing embedding

The same run with `combine="add"`, `"concat"` and `"none"` — this is the honest way
to pick between them for your data, rather than by taste.

In [ ]:
def run(combine, epochs=12, seed=0):
    set_seed(seed)
    cfg = prep.model_config(
        tokenizer=TokenizerConfig(in_channels=prep.n_features, conv_channels=32,
                                  kernel_size=5, stride=2, d_model=64),
        missing=MissingConfig(combine=combine, d_missing=16),
        transformer=TransformerConfig(n_layers=3, n_heads=8, d_ff=256, attn_mode="feature"),
    )
    t = Trainer(TSFormerRegressor(cfg),
                TrainConfig(**{**train_cfg.__dict__, "epochs": epochs,
                               "log_every_epoch": False,
                               "ckpt_path": f"best_{combine}.pt"}),
                prep.target_scaler, prep.target_names)
    h = t.fit(prep.datasets["train"], prep.datasets["holdout"], prep.datasets["oot"])
    r = t.final_report(prep.datasets)
    return {"missing": combine,
            "holdout_rmse": r.loc["holdout", "rmse"], "holdout_r2": r.loc["holdout", "r2"],
            "oot_rmse": r.loc["oot", "rmse"], "oot_r2": r.loc["oot", "r2"],
            "oot_nmad": r.loc["oot", "nmad"]}

pd.DataFrame([run(c) for c in ["none", "add", "concat"]]).set_index("missing").round(4)

## 8. Using your own data

Replace step 1 with your three tables and keep everything else:

```python
data_cfg = DataConfig(
    key="account_id",              # the primary key shared by all three tables
    time_col="month_idx",          # long format: one row per (key, time step)
    ts_features=[...],             # the C time-series columns
    target_cols=["y"],             # one or more targets
    snapshot_cat=[...], snapshot_num=[...],
    time_order="oldest_first",     # or "newest_first" if col 0 is the latest month
    systematic=SystematicMissingSpec(method="tenure", tenure_col="months_on_book"),
)
prep = prepare_datasets(ts_long, snapshot, target, data_cfg,
                        splits={"train": train_ids, "holdout": holdout_ids, "oot": oot_ids})
```

**Wide format** (`bal_1 … bal_24`, `util_1 … util_24`) — convert first:

```python
from tsfm import infer_wide_columns, wide_to_panel
cols = infer_wide_columns(wide_df.columns)                     # {feature: [(step, col), ...]}
cols = {f: [c for _, c in v] for f, v in cols.items()}
ids, X, names = wide_to_panel(wide_df, "account_id", cols)     # X: (R, C, T)
prep = prepare_datasets((ids, X), snapshot, target, data_cfg, splits)
```

`splits` can also be the name of a column in `target` holding `"train"/"holdout"/"oot"`.

### Hyper-parameters worth tuning first

| knob | why |
|---|---|
| `kernel_size` / `stride` | patch size vs. number of tokens. `k=5, s=2` over 24 months ≈ quarterly patches with overlap; `k=s=3` gives non-overlapping quarters |
| `d_model`, `n_layers`, `n_heads` | capacity — start small, the token grid is only `C × N` |
| `conv_channels` | width of the per-feature local extractor |
| `missing.combine` | `add` is cheaper and regularises; `concat` is more expressive |
| `lr`, `weight_decay`, `dropout` | the usual; transformers on tabular panels overfit fast |
| `transformer.pool` | `attention` (default), or `flatten_feature` to keep the embedding feature-attributable |

### Scaling up on GPU

* `batch_size` as large as memory allows; `amp=True` with `bf16` on Ampere or newer.
* `num_workers=4-8`, `pin_memory=True`, `persistent_workers=True`.
* `TrainConfig(compile=True)` for `torch.compile` once the geometry is settled.
* The full panel is held in CPU RAM as `float32`: `R × C × T × 4` bytes plus two
  `bool` masks. For 1M × 30 × 24 that is ~2.9 GB + 1.4 GB — use a memory-mapped
  or lazily-loaded `Dataset` beyond that.